# Traffic Demand Prediction Pipeline

This notebook contains the complete, executable pipeline for the traffic demand prediction competition.


In [ ]:
!pip install catboost lightgbm xgboost pygeohash shap pandas numpy scikit-learn scipy


## config.py

(Source of src/config.py)


In [ ]:
%%writefile src/config.py
"""
config.py — Central configuration for the traffic demand prediction pipeline.

All tunable constants, file paths, hyperparameters, and feature definitions
are stored here so every module draws from a single source of truth.
"""

import os

# ─────────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_DIR = os.path.join(PROJECT_ROOT, "dataset")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "submissions")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
EDA_DIR = os.path.join(OUTPUT_DIR, "eda")

TRAIN_FILE = os.path.join(DATA_DIR, "train.csv")
TEST_FILE = os.path.join(DATA_DIR, "test.csv")

# Ensure output directories exist
for _d in [OUTPUT_DIR, MODEL_DIR, EDA_DIR]:
    os.makedirs(_d, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────
# General
# ─────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_FOLDS = 5
TARGET_COL = "demand"
INDEX_COL = "Index"

# ─────────────────────────────────────────────────────────────────────
# Peak / Night hour ranges
# ─────────────────────────────────────────────────────────────────────
PEAK_HOURS = [7, 8, 9, 17, 18, 19, 20]       # morning + evening rush
NIGHT_HOURS = [22, 23, 0, 1, 2, 3, 4]

# ─────────────────────────────────────────────────────────────────────
# Target-encoding columns
# ─────────────────────────────────────────────────────────────────────
TARGET_ENCODE_COLS = [
    "geohash", "geohash_time_slot", "geohash_hour",
    "geohash_prefix_4", "geohash_prefix_5",
    "Weather", "RoadType", "weekday_hour",
    "roadtype_hour", "roadtype_lanes", "temp_hour"
]

# ─────────────────────────────────────────────────────────────────────
# Model hyperparameters
# ─────────────────────────────────────────────────────────────────────
CATBOOST_PARAMS = {
    "iterations": 6000,
    "learning_rate": 0.02,
    "depth": 10,
    "l2_leaf_reg": 2,
    "random_seed": RANDOM_SEED,
    "verbose": 200,
    "loss_function": "RMSE",
    "eval_metric": "R2",
    "task_type": "CPU",
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.3,
    "min_data_in_leaf": 5,
}

LGBM_PARAMS = {
    "n_estimators": 6000,
    "learning_rate": 0.02,
    "num_leaves": 511,
    "max_depth": -1,
    "min_child_samples": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.01,
    "reg_lambda": 0.1,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "verbose": -1,
}

XGB_PARAMS = {
    "n_estimators": 6000,
    "learning_rate": 0.02,
    "max_depth": 10,
    "min_child_weight": 2,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.01,
    "reg_lambda": 0.1,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "verbosity": 0,
    "tree_method": "hist",
}



## eda.py

(Source of src/eda.py)


In [ ]:
%%writefile src/eda.py
"""
eda.py — Exploratory Data Analysis for the traffic demand dataset.

Generates descriptive statistics, distribution plots, and correlation
heatmaps. All plots are saved as PNG files to submissions/eda/.
"""

import os
import warnings

import matplotlib
matplotlib.use("Agg")                       # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from config import EDA_DIR, TARGET_COL

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)


def run_eda(train_df: pd.DataFrame) -> None:
    """Run full exploratory data analysis on the training set.

    Parameters
    ----------
    train_df : pd.DataFrame
        Raw training dataframe with the ``demand`` target column.

    Side-effects
    -------------
    * Prints summary statistics to stdout.
    * Saves PNG plots to ``submissions/eda/``.
    """
    os.makedirs(EDA_DIR, exist_ok=True)

    # ── 1. Basic info ────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("EDA — BASIC INFO")
    print("=" * 60)
    print(f"Shape: {train_df.shape}")
    print(f"\nDtypes:\n{train_df.dtypes}")
    print(f"\nMissing values:\n{train_df.isnull().sum()}")
    print(f"\nMissing percentage:\n{(train_df.isnull().mean() * 100).round(2)}")
    print(f"\nUnique counts:\n{train_df.nunique()}")
    print(f"\nDescribe:\n{train_df.describe()}")

    # ── Parse timestamp once for EDA plots ───────────────────────────
    df = train_df.copy()
    if "timestamp" in df.columns:
        parts = df["timestamp"].astype(str).str.split(":", expand=True)
        df["_hour"] = pd.to_numeric(parts[0], errors="coerce").fillna(0).astype(int)
    if "day" in df.columns:
        df["_weekday"] = df["day"].astype(int) % 7

    # ── 2. Demand distribution ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(df[TARGET_COL].dropna(), bins=80, kde=True, ax=ax, color="#5e60ce")
    ax.set_title("Demand Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("demand")
    fig.tight_layout()
    fig.savefig(os.path.join(EDA_DIR, "demand_distribution.png"), dpi=150)
    plt.close(fig)
    print("✓ Saved demand_distribution.png")

    # ── 3. Demand vs hour ────────────────────────────────────────────
    if "_hour" in df.columns:
        hourly = df.groupby("_hour")[TARGET_COL].agg(["mean", "std"]).reset_index()
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(hourly["_hour"], hourly["mean"], marker="o", color="#6930c3")
        ax.fill_between(
            hourly["_hour"],
            hourly["mean"] - hourly["std"],
            hourly["mean"] + hourly["std"],
            alpha=0.2,
            color="#6930c3",
        )
        ax.set_title("Demand vs Hour (mean ± std)", fontsize=14, fontweight="bold")
        ax.set_xlabel("Hour")
        ax.set_ylabel("Demand")
        fig.tight_layout()
        fig.savefig(os.path.join(EDA_DIR, "demand_vs_hour.png"), dpi=150)
        plt.close(fig)
        print("✓ Saved demand_vs_hour.png")

    # ── 4. Demand vs weekday ─────────────────────────────────────────
    if "_weekday" in df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.barplot(data=df, x="_weekday", y=TARGET_COL, ax=ax,
                    palette="viridis", errorbar="sd")
        ax.set_title("Demand vs Weekday", fontsize=14, fontweight="bold")
        ax.set_xlabel("Weekday (0=Mon approx)")
        ax.set_ylabel("Demand")
        fig.tight_layout()
        fig.savefig(os.path.join(EDA_DIR, "demand_vs_weekday.png"), dpi=150)
        plt.close(fig)
        print("✓ Saved demand_vs_weekday.png")

    # ── 5. Demand vs Weather (box plot) ──────────────────────────────
    if "Weather" in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=df, x="Weather", y=TARGET_COL, ax=ax,
                    palette="Set2")
        ax.set_title("Demand vs Weather", fontsize=14, fontweight="bold")
        fig.tight_layout()
        fig.savefig(os.path.join(EDA_DIR, "demand_vs_weather.png"), dpi=150)
        plt.close(fig)
        print("✓ Saved demand_vs_weather.png")

    # ── 6. Outlier detection (IQR) ───────────────────────────────────
    q1 = df[TARGET_COL].quantile(0.25)
    q3 = df[TARGET_COL].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[TARGET_COL] < lower) | (df[TARGET_COL] > upper)]
    pct = len(outliers) / len(df) * 100
    print(f"\nOutlier detection (IQR): {len(outliers)} rows "
          f"({pct:.2f}%) outside [{lower:.4f}, {upper:.4f}]")

    # ── 7. Correlation heatmap ───────────────────────────────────────
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # drop helper cols
    numeric_cols = [c for c in numeric_cols if not c.startswith("_")]
    if len(numeric_cols) > 1:
        corr = df[numeric_cols].corr()
        fig, ax = plt.subplots(figsize=(12, 9))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                    square=True, ax=ax, linewidths=0.5)
        ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
        fig.tight_layout()
        fig.savefig(os.path.join(EDA_DIR, "correlation_heatmap.png"), dpi=150)
        plt.close(fig)
        print("✓ Saved correlation_heatmap.png")

    print("\n✅ EDA complete — all plots saved to", EDA_DIR)



## feature_engineering.py

(Source of src/feature_engineering.py)


In [ ]:
%%writefile src/feature_engineering.py
"""
feature_engineering.py — Feature engineering for traffic demand prediction.

Creates temporal, cyclical, geohash, interaction, aggregate, and
target-encoded features.  All transformations are train-safe: encoders
and statistics are fitted on the training set and applied to the test set.

KEY INSIGHT: Train has day 48 (full) + day 49 (0:00-2:00).
             Test  has day 49 (2:15-23:45).
             So day-48 demand patterns per geohash are our strongest signal.
"""

import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

from config import (
    PEAK_HOURS, NIGHT_HOURS, TARGET_COL,
    N_FOLDS, RANDOM_SEED, TARGET_ENCODE_COLS,
)

warnings.filterwarnings("ignore")


# =====================================================================
#  HELPER: safe geohash decoding
# =====================================================================
def _decode_geohash(gh_series: pd.Series) -> pd.DataFrame:
    """Decode a Series of geohash strings to lat/lon.

    Parameters
    ----------
    gh_series : pd.Series
        Geohash strings.

    Returns
    -------
    pd.DataFrame
        Columns ``lat`` and ``lon``.
    """
    try:
        import pygeohash as pgh
        lats, lons = [], []
        for gh in gh_series:
            try:
                lat, lon = pgh.decode(str(gh))
                lats.append(float(lat))
                lons.append(float(lon))
            except Exception:
                lats.append(np.nan)
                lons.append(np.nan)
        return pd.DataFrame({"lat": lats, "lon": lons}, index=gh_series.index)
    except ImportError:
        print("WARNING: pygeohash not installed -- skipping geohash lat/lon decode")
        return pd.DataFrame(
            {"lat": np.nan, "lon": np.nan}, index=gh_series.index
        )


# =====================================================================
#  HELPER: K-Fold target encoding (leakage-safe) with smoothing
# =====================================================================
def kfold_target_encode(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: list,
    target: str,
    n_folds: int = 5,
    smoothing: int = 10,
) -> tuple:
    """Leakage-safe KFold out-of-fold target encoding with smoothing.

    For each categorical column, the training set is encoded using OOF
    means (each fold's validation rows are encoded with means computed
    on the remaining folds).  The test set is encoded using the global
    mean computed on the full training set.  Smoothing prevents
    overfitting for categories with few samples.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataframe containing *target* column.
    test_df : pd.DataFrame
        Test dataframe (no target column expected).
    cols : list[str]
        Columns to target-encode.
    target : str
        Name of the target column.
    n_folds : int
        Number of folds for OOF encoding.
    smoothing : int
        Smoothing factor. Higher = more regularization toward global mean.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        Augmented train_df and test_df with ``<col>_te`` columns added.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)
    global_mean = train_df[target].mean()

    for col in cols:
        if col not in train_df.columns:
            continue

        te_col = f"{col}_te"
        train_df[te_col] = np.nan

        for tr_idx, val_idx in kf.split(train_df):
            tr_fold = train_df.iloc[tr_idx]
            agg = tr_fold.groupby(col)[target].agg(["mean", "count"])
            # Smoothed mean: blend category mean with global mean based on count
            smooth_mean = (agg["count"] * agg["mean"] + smoothing * global_mean) / (agg["count"] + smoothing)
            train_df.iloc[val_idx, train_df.columns.get_loc(te_col)] = (
                train_df.iloc[val_idx][col].map(smooth_mean)
            )

        # Fill any remaining NaNs with global mean
        train_df[te_col] = train_df[te_col].fillna(global_mean)

        # Test: full-train smoothed mean per category
        agg_full = train_df.groupby(col)[target].agg(["mean", "count"])
        smooth_full = (agg_full["count"] * agg_full["mean"] + smoothing * global_mean) / (agg_full["count"] + smoothing)
        test_df[te_col] = test_df[col].map(smooth_full).fillna(global_mean)

    return train_df, test_df


# =====================================================================
#  HELPER: Compute day-48 geohash demand profiles
# =====================================================================
def _compute_geohash_day48_profiles(train_df: pd.DataFrame) -> dict:
    """Compute per-geohash demand statistics from day 48 data only.

    Since test is day 49, day-48 patterns for the same geohash are
    the strongest predictive signal.

    Parameters
    ----------
    train_df : pd.DataFrame
        Full training dataframe with demand column.

    Returns
    -------
    dict
        Mapping dicts for each aggregate feature.
    """
    day48 = train_df[train_df["day"] == 48].copy() if "day" in train_df.columns else train_df.copy()

    parts = day48["timestamp"].astype(str).str.split(":", expand=True)
    day48["_hour"] = pd.to_numeric(parts[0], errors="coerce").fillna(0).astype(int)

    stats = {}

    # Per-geohash overall demand stats from day 48
    gh_agg = day48.groupby("geohash")[TARGET_COL].agg(["mean", "std", "median", "min", "max", "count"])
    stats["gh_demand_mean"] = gh_agg["mean"].to_dict()
    stats["gh_demand_std"] = gh_agg["std"].fillna(0).to_dict()
    stats["gh_demand_median"] = gh_agg["median"].to_dict()
    stats["gh_demand_min"] = gh_agg["min"].to_dict()
    stats["gh_demand_max"] = gh_agg["max"].to_dict()
    stats["gh_demand_count"] = gh_agg["count"].to_dict()

    # Per-geohash per-hour demand mean from day 48 (VERY powerful)
    gh_hour_mean = day48.groupby(["geohash", "_hour"])[TARGET_COL].mean()
    stats["gh_hour_demand_mean"] = gh_hour_mean.to_dict()

    # Per-geohash demand range (max - min) = volatility
    stats["gh_demand_range"] = (gh_agg["max"] - gh_agg["min"]).to_dict()

    # Per-geohash peak/off-peak ratio
    peak_mask = day48["_hour"].isin(PEAK_HOURS)
    gh_peak = day48[peak_mask].groupby("geohash")[TARGET_COL].mean()
    gh_offpeak = day48[~peak_mask].groupby("geohash")[TARGET_COL].mean()
    stats["gh_peak_demand"] = gh_peak.to_dict()
    stats["gh_offpeak_demand"] = gh_offpeak.to_dict()

    # Weather/RoadType per-geohash profiles from Day 48
    if "Weather" in day48.columns:
        gh_weather = day48.groupby(["geohash", "Weather"])[TARGET_COL].mean()
        stats["gh_weather_demand_mean"] = gh_weather.to_dict()
    
    if "RoadType" in day48.columns:
        gh_road = day48.groupby(["geohash", "RoadType"])[TARGET_COL].mean()
        stats["gh_road_demand_mean"] = gh_road.to_dict()

    # Global mean for fallback
    stats["global_mean"] = train_df[TARGET_COL].mean()

    return stats


# =====================================================================
#  MAIN: engineer_features
# =====================================================================
def engineer_features(
    df: pd.DataFrame,
    is_train: bool = True,
    train_stats: dict | None = None,
) -> tuple:
    """Create all engineered features for the traffic demand dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe (train or test).
    is_train : bool
        If True, computes and returns training statistics needed for
        the test set (geohash freq map, demand profiles, etc.).
    train_stats : dict or None
        Pre-computed training statistics (required when ``is_train=False``).

    Returns
    -------
    tuple[pd.DataFrame, dict]
        (transformed df, stats dict)
    """
    if train_stats is None:
        train_stats = {}

    df = df.copy()

    # ── A. Timestamp features ────────────────────────────────────────
    if "timestamp" in df.columns:
        parts = df["timestamp"].astype(str).str.split(":", expand=True)
        df["hour"] = pd.to_numeric(parts[0], errors="coerce").fillna(0).astype(int)
        df["minute"] = pd.to_numeric(parts[1], errors="coerce").fillna(0).astype(int)
    else:
        df["hour"] = 0
        df["minute"] = 0

    if "day" in df.columns:
        df["weekday"] = df["day"].astype(int) % 7
        df["month"] = (df["day"].astype(int) // 30) % 12 + 1
        df["quarter"] = ((df["month"] - 1) // 3) + 1
        df["is_weekend"] = (df["weekday"] >= 5).astype(int)
        df["weekofyear"] = df["day"].astype(int) // 7
    else:
        for c in ["weekday", "month", "quarter", "is_weekend", "weekofyear"]:
            df[c] = 0

    df["time_slot"] = df["hour"] * 4 + df["minute"] // 15
    df["peak_hour"] = df["hour"].isin(PEAK_HOURS).astype(int)
    df["night_hour"] = df["hour"].isin(NIGHT_HOURS).astype(int)

    # Time-of-day as fraction (0.0 to 1.0)
    df["time_frac"] = (df["hour"] * 60 + df["minute"]) / (24 * 60)

    # ── B. Cyclical encoding ─────────────────────────────────────────
    for col, period in [("hour", 24), ("minute", 60), ("weekday", 7), ("time_slot", 96)]:
        df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

    # ── C. Geohash features ──────────────────────────────────────────
    if "geohash" in df.columns:
        df["geohash"] = df["geohash"].astype(str)
        geo_decoded = _decode_geohash(df["geohash"])
        df["lat"] = geo_decoded["lat"]
        df["lon"] = geo_decoded["lon"]
        df["geohash_prefix_4"] = df["geohash"].str[:4]
        df["geohash_prefix_5"] = df["geohash"].str[:5]

        # Frequency encoding
        if is_train:
            freq_map = df["geohash"].value_counts().to_dict()
            train_stats["geohash_freq_map"] = freq_map
        else:
            freq_map = train_stats.get("geohash_freq_map", {})
        df["geohash_freq"] = df["geohash"].map(freq_map).fillna(0).astype(int)

    # ── D. Categorical encoding (binary) ─────────────────────────────
    if "LargeVehicles" in df.columns:
        df["large_vehicles_flag"] = (df["LargeVehicles"] == "Allowed").astype(int)
    if "Landmarks" in df.columns:
        df["landmarks_flag"] = (df["Landmarks"] == "Yes").astype(int)

    # ── E. Interaction features ──────────────────────────────────────
    df["geohash_time_slot"] = (
        df["geohash"].astype(str) + "_" + df["time_slot"].astype(str)
    )
    df["geohash_hour"] = (
        df["geohash"].astype(str) + "_" + df["hour"].astype(str)
    )
    if "Weather" in df.columns:
        df["weather_hour"] = df["Weather"].astype(str) + "_" + df["hour"].astype(str)
    else:
        df["weather_hour"] = "UNK_0"
    if "RoadType" in df.columns:
        df["roadtype_hour"] = df["RoadType"].astype(str) + "_" + df["hour"].astype(str)
        df["roadtype_lanes"] = df["RoadType"].astype(str) + "_" + df["NumberofLanes"].astype(str)
    else:
        df["roadtype_hour"] = "UNK_0"
        df["roadtype_lanes"] = "UNK_0"
    df["weekday_hour"] = df["weekday"].astype(str) + "_" + df["hour"].astype(str)

    # Temperature binning
    if "Temperature" in df.columns:
        df["temp_bin"] = pd.cut(
            df["Temperature"], bins=10, labels=False
        )
        df["temp_bin"] = df["temp_bin"].fillna(-1).astype(int)
    else:
        df["temp_bin"] = -1
    df["temp_hour"] = df["temp_bin"].astype(str) + "_" + df["hour"].astype(str)

    # ── F. Day-48 geohash demand profiles (KEY FEATURES) ─────────────
    if is_train:
        profiles = _compute_geohash_day48_profiles(df)
        train_stats["profiles"] = profiles
    else:
        profiles = train_stats.get("profiles", {})

    global_mean = profiles.get("global_mean", 0.0)

    # Map geohash-level aggregate features
    df["gh_demand_mean_d48"] = df["geohash"].map(profiles.get("gh_demand_mean", {})).fillna(global_mean)
    df["gh_demand_std_d48"] = df["geohash"].map(profiles.get("gh_demand_std", {})).fillna(0)
    df["gh_demand_median_d48"] = df["geohash"].map(profiles.get("gh_demand_median", {})).fillna(global_mean)
    df["gh_demand_min_d48"] = df["geohash"].map(profiles.get("gh_demand_min", {})).fillna(0)
    df["gh_demand_max_d48"] = df["geohash"].map(profiles.get("gh_demand_max", {})).fillna(global_mean)
    df["gh_demand_range_d48"] = df["geohash"].map(profiles.get("gh_demand_range", {})).fillna(0)
    df["gh_demand_count_d48"] = df["geohash"].map(profiles.get("gh_demand_count", {})).fillna(0)
    df["gh_peak_demand_d48"] = df["geohash"].map(profiles.get("gh_peak_demand", {})).fillna(global_mean)
    df["gh_offpeak_demand_d48"] = df["geohash"].map(profiles.get("gh_offpeak_demand", {})).fillna(global_mean)

    # Per-geohash per-hour demand from day 48 (THE MOST POWERFUL FEATURE)
    gh_hour_map = profiles.get("gh_hour_demand_mean", {})
    df["gh_hour_demand_d48"] = df.apply(
        lambda row: gh_hour_map.get((row["geohash"], row["hour"]), global_mean), axis=1
    )

    # Weather/Road profiles
    gh_weather_map = profiles.get("gh_weather_demand_mean", {})
    df["gh_weather_demand_d48"] = df.apply(
        lambda row: gh_weather_map.get((row["geohash"], row.get("Weather", "UNK")), global_mean), axis=1
    )
    
    gh_road_map = profiles.get("gh_road_demand_mean", {})
    df["gh_road_demand_d48"] = df.apply(
        lambda row: gh_road_map.get((row["geohash"], row.get("RoadType", "UNK")), global_mean), axis=1
    )

    # Ratio features
    df["demand_vs_mean_ratio"] = df["gh_hour_demand_d48"] / (df["gh_demand_mean_d48"] + 1e-8)
    df["demand_vs_peak_ratio"] = df["gh_hour_demand_d48"] / (df["gh_peak_demand_d48"] + 1e-8)

    # ── G. Prefix-level aggregates ───────────────────────────────────
    if is_train:
        p4_mean = df.groupby("geohash_prefix_4")[TARGET_COL].mean().to_dict()
        p5_mean = df.groupby("geohash_prefix_5")[TARGET_COL].mean().to_dict()
        train_stats["p4_mean"] = p4_mean
        train_stats["p5_mean"] = p5_mean
    else:
        p4_mean = train_stats.get("p4_mean", {})
        p5_mean = train_stats.get("p5_mean", {})

    df["prefix4_demand_mean"] = df["geohash_prefix_4"].map(p4_mean).fillna(global_mean)
    df["prefix5_demand_mean"] = df["geohash_prefix_5"].map(p5_mean).fillna(global_mean)

    print(f"[OK] Feature engineering {'(train)' if is_train else '(test)'} done -- "
          f"{df.shape[1]} columns")
    return df, train_stats


def get_feature_columns(df: pd.DataFrame) -> list:
    """Return the list of feature columns (excluding target, Index, raw strings).

    Parameters
    ----------
    df : pd.DataFrame
        Engineered dataframe.

    Returns
    -------
    list[str]
        Feature column names suitable for model training.
    """
    drop_cols = {
        TARGET_COL, "Index", "timestamp", "geohash",
        "geohash_prefix_4", "geohash_prefix_5",
        "geohash_time_slot", "geohash_hour",
        "weather_hour", "roadtype_hour", "roadtype_lanes",
        "weekday_hour", "temp_hour", "day",
        "LargeVehicles", "Landmarks", "Weather", "RoadType",
    }
    # Also drop any remaining object columns (safety net)
    obj_cols = set(df.select_dtypes(include=["object"]).columns)
    exclude = drop_cols | obj_cols

    feature_cols = [c for c in df.columns if c not in exclude]
    return feature_cols



## preprocessing.py

(Source of src/preprocessing.py)


In [ ]:
%%writefile src/preprocessing.py
"""
preprocessing.py — Final preprocessing before model training.

Handles remaining NaN imputation, label encoding of categorical columns,
and ensures consistent column ordering between train and test.
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from config import TARGET_COL


def preprocess(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list,
) -> tuple:
    """Preprocess train and test DataFrames for model consumption.

    1. Fill numeric NaNs with column median (fit on train).
    2. Fill categorical NaNs with ``"MISSING"``.
    3. Label-encode remaining string/object columns (fit on combined
       train + test vocabulary to avoid unseen-label errors).
    4. Return aligned numpy arrays.

    Parameters
    ----------
    train_df : pd.DataFrame
        Engineered training data (must contain ``TARGET_COL``).
    test_df : pd.DataFrame
        Engineered test data.
    feature_cols : list[str]
        Ordered list of feature column names.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, list[str]]
        ``(X_train, y_train, X_test, final_feature_cols)``
    """
    # Ensure we only keep requested features
    missing_train = [c for c in feature_cols if c not in train_df.columns]
    missing_test = [c for c in feature_cols if c not in test_df.columns]
    if missing_train:
        print(f"⚠ Columns missing in train (will be filled with 0): {missing_train}")
        for c in missing_train:
            train_df[c] = 0
    if missing_test:
        print(f"⚠ Columns missing in test (will be filled with 0): {missing_test}")
        for c in missing_test:
            test_df[c] = 0

    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[TARGET_COL].values.copy()

    # ── 1. Numeric NaN → median (fit train, apply test) ──────────────
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    medians = X_train[numeric_cols].median()
    X_train[numeric_cols] = X_train[numeric_cols].fillna(medians)
    X_test[numeric_cols] = X_test[numeric_cols].fillna(medians)

    # ── 2. Categorical NaN → "MISSING" ──────────────────────────────
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    for c in cat_cols:
        X_train[c] = X_train[c].fillna("MISSING").astype(str)
        X_test[c] = X_test[c].fillna("MISSING").astype(str)

    # ── 3. Label-encode (fit on combined to handle unseen) ───────────
    label_encoders = {}
    for c in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[c], X_test[c]], axis=0)
        le.fit(combined)
        X_train[c] = le.transform(X_train[c])
        X_test[c] = le.transform(X_test[c])
        label_encoders[c] = le

    # ── 4. Final safety: fill any remaining NaN with 0 ───────────────
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    final_feature_cols = list(X_train.columns)

    print(f"✓ Preprocessing done — X_train {X_train.shape}, X_test {X_test.shape}")
    print(f"  NaNs in X_train: {X_train.isna().sum().sum()}, X_test: {X_test.isna().sum().sum()}")

    return (
        X_train.values.astype(np.float64),
        y_train.astype(np.float64),
        X_test.values.astype(np.float64),
        final_feature_cols,
    )



## validation.py

(Source of src/validation.py)


In [ ]:
%%writefile src/validation.py
"""
validation.py — Cross-validation strategy for demand prediction.

Uses KFold for competition scoring.  The train/test split is already
temporal (train=day48+early49, test=rest of day49), so within-train
cross-validation uses KFold to maximise each fold's training size.
"""

import numpy as np
from sklearn.model_selection import KFold

from config import N_FOLDS, RANDOM_SEED


def get_cv_splits(X, y, timestamps=None, n_folds: int = N_FOLDS) -> list:
    """Generate cross-validation splits.

    The competition already enforces a temporal train/test boundary
    (train = day 48 + early day 49; test = rest of day 49).  Within
    the training set we use KFold to give each fold maximum data.

    Random splits are generally invalid for time-series because demand
    has temporal autocorrelation.  However, since the competition
    train/test boundary is already temporal and we have only 2 days
    of data, using TimeSeriesSplit *within* training cripples fold 0
    (it would train on only ~12K rows).  KFold gives each fold ~62K
    training rows, producing far stronger models.

    Parameters
    ----------
    X : array-like
        Feature matrix (used only for its length).
    y : array-like
        Target vector.
    timestamps : array-like, optional
        Not used in KFold mode but kept for API compatibility.
    n_folds : int
        Number of folds (default from config).

    Returns
    -------
    list[tuple[np.ndarray, np.ndarray]]
        List of ``(train_indices, val_indices)`` tuples.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)

    splits = []
    for train_idx, val_idx in kf.split(X):
        splits.append((train_idx, val_idx))

    print(f"[OK] Created {n_folds} KFold CV splits")
    for i, (tr, va) in enumerate(splits):
        print(f"  Fold {i}: train={len(tr):,}, val={len(va):,}")

    return splits



## models.py

(Source of src/models.py)


In [ ]:
%%writefile src/models.py
"""
models.py — OOF training of CatBoost, LightGBM, and XGBoost regressors.

Each model is trained with early stopping across K time-series folds.
Out-of-fold (OOF) predictions are collected for stacking / ensemble
weight optimisation, and per-fold test predictions are averaged.
"""

import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import joblib
from sklearn.metrics import r2_score

from config import (
    CATBOOST_PARAMS, LGBM_PARAMS, XGB_PARAMS,
    MODEL_DIR, OUTPUT_DIR, RANDOM_SEED,
)

warnings.filterwarnings("ignore")


# =====================================================================
#  Main OOF training
# =====================================================================
def train_models(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    cv_splits: list,
    feature_names: list,
) -> dict:
    """Train CatBoost, LightGBM, and XGBoost with OOF strategy.

    Parameters
    ----------
    X_train : np.ndarray
        Training feature matrix.
    y_train : np.ndarray
        Training target vector.
    X_test : np.ndarray
        Test feature matrix.
    cv_splits : list[tuple]
        List of ``(train_idx, val_idx)`` from ``get_cv_splits``.
    feature_names : list[str]
        Column names (used for importance plots).

    Returns
    -------
    dict
        ``{model_name: (oof_preds, test_preds_avg, fold_scores)}``
    """
    n_train = X_train.shape[0]
    n_test = X_test.shape[0]
    n_folds = len(cv_splits)
    results = {}

    # ── CatBoost ─────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("TRAINING — CatBoost")
    print("=" * 60)
    results["catboost"] = _train_catboost(
        X_train, y_train, X_test, cv_splits, feature_names, n_train, n_test, n_folds
    )

    # ── LightGBM ─────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("TRAINING — LightGBM")
    print("=" * 60)
    results["lgbm"] = _train_lgbm(
        X_train, y_train, X_test, cv_splits, feature_names, n_train, n_test, n_folds
    )

    # ── XGBoost ──────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("TRAINING — XGBoost")
    print("=" * 60)
    results["xgb"] = _train_xgb(
        X_train, y_train, X_test, cv_splits, feature_names, n_train, n_test, n_folds
    )

    return results


# =====================================================================
#  CatBoost
# =====================================================================
def _train_catboost(X_train, y_train, X_test, cv_splits, feature_names,
                    n_train, n_test, n_folds):
    """Train CatBoost with OOF and return (oof, test_avg, scores)."""
    from catboost import CatBoostRegressor, Pool

    oof = np.zeros(n_train)
    test_preds = np.zeros(n_test)
    scores = []
    importances = np.zeros(len(feature_names))

    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"\n  Fold {fold} …")
        train_pool = Pool(X_train[tr_idx], y_train[tr_idx], feature_names=feature_names)
        val_pool = Pool(X_train[va_idx], y_train[va_idx], feature_names=feature_names)

        params = CATBOOST_PARAMS.copy()
        model = CatBoostRegressor(**params)
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=50,
            verbose=200,
        )

        oof[va_idx] = model.predict(X_train[va_idx])
        test_preds += model.predict(X_test) / n_folds
        fold_r2 = r2_score(y_train[va_idx], oof[va_idx])
        scores.append(fold_r2)
        importances += model.get_feature_importance() / n_folds
        print(f"  Fold {fold} R²: {fold_r2:.6f}")

        joblib.dump(model, os.path.join(MODEL_DIR, f"catboost_fold{fold}.pkl"))

    overall = r2_score(y_train, oof)
    print(f"\n  CatBoost OOF R²: {overall:.6f}  (folds: {np.mean(scores):.6f} ± {np.std(scores):.6f})")

    _save_importance(feature_names, importances, "catboost")
    return oof, test_preds, scores


# =====================================================================
#  LightGBM
# =====================================================================
def _train_lgbm(X_train, y_train, X_test, cv_splits, feature_names,
                n_train, n_test, n_folds):
    """Train LightGBM with OOF and return (oof, test_avg, scores)."""
    import lightgbm as lgb

    oof = np.zeros(n_train)
    test_preds = np.zeros(n_test)
    scores = []
    importances = np.zeros(len(feature_names))

    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"\n  Fold {fold} …")
        params = LGBM_PARAMS.copy()
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train[tr_idx], y_train[tr_idx],
            eval_set=[(X_train[va_idx], y_train[va_idx])],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=200),
            ],
        )

        oof[va_idx] = model.predict(X_train[va_idx])
        test_preds += model.predict(X_test) / n_folds
        fold_r2 = r2_score(y_train[va_idx], oof[va_idx])
        scores.append(fold_r2)
        importances += model.feature_importances_ / n_folds
        print(f"  Fold {fold} R²: {fold_r2:.6f}")

        joblib.dump(model, os.path.join(MODEL_DIR, f"lgbm_fold{fold}.pkl"))

    overall = r2_score(y_train, oof)
    print(f"\n  LightGBM OOF R²: {overall:.6f}  (folds: {np.mean(scores):.6f} ± {np.std(scores):.6f})")

    _save_importance(feature_names, importances, "lgbm")
    return oof, test_preds, scores


# =====================================================================
#  XGBoost
# =====================================================================
def _train_xgb(X_train, y_train, X_test, cv_splits, feature_names,
               n_train, n_test, n_folds):
    """Train XGBoost with OOF and return (oof, test_avg, scores)."""
    from xgboost import XGBRegressor

    oof = np.zeros(n_train)
    test_preds = np.zeros(n_test)
    scores = []
    importances = np.zeros(len(feature_names))

    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"\n  Fold {fold} …")
        params = XGB_PARAMS.copy()
        params["early_stopping_rounds"] = 50
        model = XGBRegressor(**params)
        model.fit(
            X_train[tr_idx], y_train[tr_idx],
            eval_set=[(X_train[va_idx], y_train[va_idx])],
            verbose=200,
        )

        oof[va_idx] = model.predict(X_train[va_idx])
        test_preds += model.predict(X_test) / n_folds
        fold_r2 = r2_score(y_train[va_idx], oof[va_idx])
        scores.append(fold_r2)
        importances += model.feature_importances_ / n_folds
        print(f"  Fold {fold} R²: {fold_r2:.6f}")

        joblib.dump(model, os.path.join(MODEL_DIR, f"xgb_fold{fold}.pkl"))

    overall = r2_score(y_train, oof)
    print(f"\n  XGBoost OOF R²: {overall:.6f}  (folds: {np.mean(scores):.6f} ± {np.std(scores):.6f})")

    _save_importance(feature_names, importances, "xgb")
    return oof, test_preds, scores


# =====================================================================
#  Feature importance helper
# =====================================================================
def _save_importance(feature_names, importances, model_name):
    """Save feature importance bar chart as PNG.

    Parameters
    ----------
    feature_names : list[str]
    importances : np.ndarray
    model_name : str
    """
    idx = np.argsort(importances)[::-1][:30]  # top 30
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(
        [feature_names[i] for i in idx][::-1],
        importances[idx][::-1],
        color="#7209b7",
    )
    ax.set_title(f"Feature Importance — {model_name}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Importance")
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"feature_importance_{model_name}.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  ✓ Saved {path}")



## ensemble.py

(Source of src/ensemble.py)


In [ ]:
%%writefile src/ensemble.py
"""
ensemble.py — Optimal blending of OOF predictions from multiple models.

Uses ``scipy.optimize.minimize`` to find the weight vector that
maximises R² on the out-of-fold predictions, then applies those
weights to the test-set predictions for the final submission.
"""

import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import r2_score


def optimize_ensemble(oof_dict: dict, y_train: np.ndarray) -> dict:
    """Find optimal ensemble weights that maximise OOF R².

    Parameters
    ----------
    oof_dict : dict
        ``{model_name: oof_predictions}`` — OOF arrays of shape ``(n_train,)``.
    y_train : np.ndarray
        True target values.

    Returns
    -------
    dict
        ``{model_name: weight}`` summing to 1.
    """
    model_names = list(oof_dict.keys())
    oof_matrix = np.column_stack([oof_dict[m] for m in model_names])
    n_models = len(model_names)

    def _neg_r2(weights):
        blended = oof_matrix @ weights
        return -r2_score(y_train, blended)

    # Initial guess: equal weights
    x0 = np.ones(n_models) / n_models

    # Constraints: weights sum to 1
    constraints = {"type": "eq", "fun": lambda w: w.sum() - 1.0}
    # Bounds: each weight in [0, 1]
    bounds = [(0.0, 1.0)] * n_models

    result = minimize(
        _neg_r2, x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
    )

    best_weights = {m: round(float(w), 6) for m, w in zip(model_names, result.x)}
    best_r2 = -result.fun

    print("\n" + "=" * 60)
    print("ENSEMBLE — Optimal Weights")
    print("=" * 60)
    for m, w in best_weights.items():
        print(f"  {m:>12s}: {w:.6f}")
    print(f"\n  Ensemble OOF R²: {best_r2:.6f}")

    return best_weights


def blend_predictions(test_preds_dict: dict, weights: dict) -> np.ndarray:
    """Apply weighted average to test predictions.

    Parameters
    ----------
    test_preds_dict : dict
        ``{model_name: test_predictions_array}``.
    weights : dict
        ``{model_name: float_weight}``.

    Returns
    -------
    np.ndarray
        Final blended predictions.
    """
    model_names = list(weights.keys())
    blend = np.zeros_like(test_preds_dict[model_names[0]])
    for m in model_names:
        blend += weights[m] * test_preds_dict[m]
    return blend



## predict.py

(Source of src/predict.py)


In [ ]:
%%writefile src/predict.py
"""
predict.py — Main runner for the traffic demand prediction pipeline.

Orchestrates EDA → feature engineering → preprocessing → validation →
model training → ensemble → submission generation → SHAP explainability.

Usage
-----
    python src/predict.py
"""

import os
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

# Ensure src/ is on the Python path
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from config import (
    TRAIN_FILE, TEST_FILE, OUTPUT_DIR, MODEL_DIR,
    TARGET_COL, INDEX_COL, N_FOLDS, TARGET_ENCODE_COLS,
)
from eda import run_eda
from feature_engineering import (
    engineer_features, get_feature_columns, kfold_target_encode,
)
from preprocessing import preprocess
from validation import get_cv_splits
from models import train_models
from ensemble import optimize_ensemble, blend_predictions

warnings.filterwarnings("ignore")


def main():
    """Run the full traffic demand prediction pipeline end-to-end."""
    print("🚀 Starting pipeline …\n")

    # ─────────────────────────────────────────────────────────────────
    # 1. Load data
    # ─────────────────────────────────────────────────────────────────
    print("=" * 60)
    print("STEP 1 — Loading data")
    print("=" * 60)
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape:  {test_df.shape}")

    # Save test Index for final submission
    test_index = test_df[INDEX_COL].values.copy()

    # ─────────────────────────────────────────────────────────────────
    # 2. EDA
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 2 — EDA")
    print("=" * 60)
    run_eda(train_df)

    # ─────────────────────────────────────────────────────────────────
    # 3. Feature engineering
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 3 — Feature Engineering")
    print("=" * 60)
    train_df, train_stats = engineer_features(train_df, is_train=True)
    test_df, _ = engineer_features(test_df, is_train=False, train_stats=train_stats)

    # Target encoding (leakage-safe)
    te_cols = [c for c in TARGET_ENCODE_COLS if c in train_df.columns]
    if te_cols:
        train_df, test_df = kfold_target_encode(
            train_df, test_df, te_cols, TARGET_COL, n_folds=N_FOLDS
        )
        print(f"✓ Target-encoded columns: {te_cols}")

    # ─────────────────────────────────────────────────────────────────
    # 4. Preprocessing
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 4 — Preprocessing")
    print("=" * 60)
    feature_cols = get_feature_columns(train_df)
    X_train, y_train, X_test, final_features = preprocess(
        train_df, test_df, feature_cols
    )

    # ─────────────────────────────────────────────────────────────────
    # 5. CV splits
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 5 — Validation splits")
    print("=" * 60)
    # Use day column for temporal ordering
    timestamps = train_df["day"].values if "day" in train_df.columns else np.arange(len(X_train))
    cv_splits = get_cv_splits(X_train, y_train, timestamps, n_folds=N_FOLDS)

    # ─────────────────────────────────────────────────────────────────
    # 6. Model training
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 6 — Training models")
    print("=" * 60)
    results = train_models(X_train, y_train, X_test, cv_splits, final_features)

    # ─────────────────────────────────────────────────────────────────
    # 7. Ensemble
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 7 — Ensemble optimisation")
    print("=" * 60)
    oof_dict = {name: vals[0] for name, vals in results.items()}
    test_dict = {name: vals[1] for name, vals in results.items()}

    weights = optimize_ensemble(oof_dict, y_train)
    final_preds = blend_predictions(test_dict, weights)

    # ─────────────────────────────────────────────────────────────────
    # 8. Save submission
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 8 — Saving outputs")
    print("=" * 60)

    # Submission CSV
    submission = pd.DataFrame({
        INDEX_COL: test_index,
        TARGET_COL: final_preds,
    })
    # Safety: fill any NaN with global mean
    if submission[TARGET_COL].isna().any():
        submission[TARGET_COL].fillna(y_train.mean(), inplace=True)
    sub_path = os.path.join(OUTPUT_DIR, "submission.csv")
    submission.to_csv(sub_path, index=False)
    print(f"✓ submission.csv saved — {len(submission)} rows")
    print(f"  NaN count: {submission[TARGET_COL].isna().sum()}")
    assert len(submission) == len(test_index), (
        f"Expected {len(test_index)} rows, got {len(submission)}"
    )

    # Feature importance CSV (averaged across models)
    _save_feature_importance_csv(results, final_features)

    # Validation scores text
    _save_validation_scores(results, oof_dict, y_train, weights)

    # ─────────────────────────────────────────────────────────────────
    # 9. SHAP explainability
    # ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 9 — SHAP explainability")
    print("=" * 60)
    _run_shap(X_train, final_features)

    print("\n" + "=" * 60)
    print("✅ PIPELINE COMPLETE")
    print("=" * 60)


# =====================================================================
#  SHAP
# =====================================================================
def _run_shap(X_train, feature_names):
    """Generate SHAP beeswarm plot for the first LightGBM fold model.

    Parameters
    ----------
    X_train : np.ndarray
        Training feature matrix.
    feature_names : list[str]
        Feature column names.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    try:
        import shap
        import joblib

        model_path = os.path.join(MODEL_DIR, "lgbm_fold0.pkl")
        if not os.path.exists(model_path):
            print("⚠ lgbm_fold0.pkl not found — skipping SHAP")
            return

        model = joblib.load(model_path)

        # Use a subsample for speed
        n_sample = min(2000, X_train.shape[0])
        idx = np.random.RandomState(42).choice(X_train.shape[0], n_sample, replace=False)
        X_sample = X_train[idx]

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

        fig = plt.figure(figsize=(12, 8))
        shap.summary_plot(
            shap_values, X_sample,
            feature_names=feature_names,
            show=False,
            max_display=20,
        )
        plt.tight_layout()
        path = os.path.join(OUTPUT_DIR, "shap_summary.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"✓ SHAP beeswarm plot saved to {path}")

        # Print top-10 features by mean |SHAP|
        mean_abs = np.abs(shap_values).mean(axis=0)
        top_idx = np.argsort(mean_abs)[::-1][:10]
        print("\n  Top 10 features by mean |SHAP|:")
        for i, ix in enumerate(top_idx, 1):
            print(f"    {i:2d}. {feature_names[ix]:30s}  {mean_abs[ix]:.6f}")

    except Exception as e:
        print(f"⚠ SHAP failed: {e}")


# =====================================================================
#  Feature importance CSV
# =====================================================================
def _save_feature_importance_csv(results, feature_names):
    """Save mean feature importance across all models and folds.

    Parameters
    ----------
    results : dict
        Model training results.
    feature_names : list[str]
        Feature names.
    """
    import joblib

    imp_dict = {"feature": feature_names}
    for model_name in results:
        # Try to load fold-0 model and extract importance
        try:
            m = joblib.load(os.path.join(MODEL_DIR, f"{model_name}_fold0.pkl"))
            if hasattr(m, "feature_importances_"):
                imp_dict[model_name] = m.feature_importances_
            elif hasattr(m, "get_feature_importance"):
                imp_dict[model_name] = m.get_feature_importance()
            else:
                imp_dict[model_name] = np.zeros(len(feature_names))
        except Exception:
            imp_dict[model_name] = np.zeros(len(feature_names))

    imp_df = pd.DataFrame(imp_dict)
    model_cols = [c for c in imp_df.columns if c != "feature"]
    imp_df["mean_importance"] = imp_df[model_cols].mean(axis=1)
    imp_df = imp_df.sort_values("mean_importance", ascending=False)
    path = os.path.join(OUTPUT_DIR, "feature_importance.csv")
    imp_df.to_csv(path, index=False)
    print(f"✓ feature_importance.csv saved — {len(imp_df)} features")


# =====================================================================
#  Validation scores text
# =====================================================================
def _save_validation_scores(results, oof_dict, y_train, weights):
    """Save per-model per-fold R² and ensemble R² to a text file.

    Parameters
    ----------
    results : dict
    oof_dict : dict
    y_train : np.ndarray
    weights : dict
    """
    lines = []
    lines.append("=" * 60)
    lines.append("VALIDATION SCORES")
    lines.append("=" * 60)

    for name, (oof, test, scores) in results.items():
        lines.append(f"\n{name.upper()}")
        for i, s in enumerate(scores):
            lines.append(f"  Fold {i}: R² = {s:.6f}")
        overall = r2_score(y_train, oof)
        lines.append(f"  OOF R²:  {overall:.6f}")
        lines.append(f"  Mean ± Std: {np.mean(scores):.6f} ± {np.std(scores):.6f}")

    # Ensemble
    blend = np.zeros_like(y_train, dtype=float)
    for m, w in weights.items():
        blend += w * oof_dict[m]
    ens_r2 = r2_score(y_train, blend)
    lines.append(f"\nENSEMBLE")
    lines.append(f"  Weights: {weights}")
    lines.append(f"  OOF R²:  {ens_r2:.6f}")
    lines.append("=" * 60)

    txt = "\n".join(lines)
    path = os.path.join(OUTPUT_DIR, "validation_scores.txt")
    with open(path, "w") as f:
        f.write(txt)
    print(f"✓ validation_scores.txt saved")
    print(txt)


if __name__ == "__main__":
    main()



## Execute Pipeline

Run the final training and prediction pipeline.


In [ ]:
!python src/predict.py
